[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/colab-slm-playground/blob/main/trending_gguf_chatbot_colab.ipynb)

# Trending GGUF Chatbot

Fetches trending GGUF models from Hugging Face, lets you pick a model and quantization variant, and runs a chat UI inside Colab.

Uses [`llama-cpp-python`](https://github.com/abetlen/llama-cpp-python) — runs on **CPU**, no GPU required.
Smaller quants (Q4_K_M) are faster; larger quants (Q8_0) are higher quality.

## 1 — Install dependencies

Builds `llama-cpp-python` with OpenBLAS for CPU acceleration. Takes ~2 minutes.

In [ ]:
import os
os.environ["CMAKE_ARGS"] = "-DGGML_BLAS=ON -DGGML_BLAS_VENDOR=OpenBLAS"
os.environ["FORCE_CMAKE"] = "1"

!pip install -q llama-cpp-python huggingface_hub ipywidgets

## 2 — Fetch trending models

In [ ]:
from huggingface_hub import HfApi
from IPython.display import display, HTML

print("Fetching trending GGUF text-generation models...")

api = HfApi()
raw_models = list(api.list_models(
    filter="gguf",
    pipeline_tag="text-generation",
    sort="trending_score",
    direction=-1,
    limit=30,
    full=True,
))

# Quant preference order for default selection (balance of speed vs quality)
_QUANT_RANK = ["Q4_K_M", "Q4_K_S", "Q4_0", "Q5_K_M", "Q5_K_S", "Q3_K_M", "Q8_0", "Q2_K"]

def _sort_gguf_files(files):
    def rank(f):
        upper = f.upper()
        for i, q in enumerate(_QUANT_RANK):
            if q in upper:
                return i
        return len(_QUANT_RANK)
    return sorted(files, key=rank)

model_info = []
for m in raw_models:
    gguf_files = [
        s.rfilename for s in (m.siblings or [])
        if s.rfilename.endswith(".gguf")
    ]
    if not gguf_files:
        continue
    model_info.append({
        "id": m.id,
        "downloads": m.downloads or 0,
        "likes": m.likes or 0,
        "gguf_files": _sort_gguf_files(gguf_files),
    })

rows = "".join(
    f"<tr>"
    f"<td>{i+1}</td>"
    f"<td><code>{m['id']}</code></td>"
    f"<td>{m['downloads']:,}</td>"
    f"<td>{m['likes']:,}</td>"
    f"<td>{len(m['gguf_files'])}</td>"
    f"</tr>"
    for i, m in enumerate(model_info)
)

display(HTML(f"""
<style>
  .gguf-table {{
    border-collapse: collapse;
    font-family: monospace;
    font-size: 13px;
    color-scheme: light dark;
  }}
  .gguf-table td, .gguf-table th {{
    padding: 5px 12px;
  }}
  .gguf-table th {{
    text-align: left;
    border-bottom: 2px solid;
  }}
  .gguf-table td:nth-child(3),
  .gguf-table td:nth-child(4),
  .gguf-table td:nth-child(5),
  .gguf-table th:nth-child(3),
  .gguf-table th:nth-child(4),
  .gguf-table th:nth-child(5) {{
    text-align: right;
  }}
  @media (prefers-color-scheme: light) {{
    .gguf-table th {{ background: #f0f0f0; border-color: #bbb; color: #111; }}
    .gguf-table td {{ color: #111; }}
    .gguf-table tr:hover td {{ background: #f7f7f7; }}
  }}
  @media (prefers-color-scheme: dark) {{
    .gguf-table th {{ background: #2a2a2a; border-color: #555; color: #ddd; }}
    .gguf-table td {{ color: #ccc; }}
    .gguf-table tr:hover td {{ background: #1e1e1e; }}
  }}
</style>
<table class="gguf-table">
  <thead>
    <tr>
      <th>#</th>
      <th>Model ID</th>
      <th>Downloads</th>
      <th>Likes</th>
      <th>Variants</th>
    </tr>
  </thead>
  <tbody>{rows}</tbody>
</table>
"""))

print(f"Found {len(model_info)} models with GGUF files.")

## 3 — Select model & quantization variant

Pick a model, then pick a quant. The file dropdown updates automatically.
When ready, click **Load Model**.

In [ ]:
import os
import ipywidgets as widgets
from llama_cpp import Llama
from IPython.display import display

llm = None
loaded_model_id = None

model_files = {m['id']: m['gguf_files'] for m in model_info}
first_model = model_info[0]

model_dropdown = widgets.Dropdown(
    options=[m['id'] for m in model_info],
    description="Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="620px"),
)

file_dropdown = widgets.Dropdown(
    options=first_model['gguf_files'],
    description="Quant:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="620px"),
)

def on_model_change(change):
    files = model_files.get(change['new'], [])
    file_dropdown.options = files
    if files:
        file_dropdown.value = files[0]

model_dropdown.observe(on_model_change, names='value')

load_btn = widgets.Button(
    description="Load Model",
    button_style="primary",
    layout=widgets.Layout(width="140px", height="36px"),
)

status = widgets.Output()

def on_load(b):
    global llm, loaded_model_id
    selected_id   = model_dropdown.value
    selected_file = file_dropdown.value
    load_btn.disabled = True
    load_btn.description = "Loading..."
    with status:
        status.clear_output()
        print(f"Loading : {selected_id}")
        print(f"Variant : {selected_file}")
        print("Downloading — this may take a few minutes...")
        try:
            llm = Llama.from_pretrained(
                repo_id=selected_id,
                filename=selected_file,
                n_ctx=4096,
                n_threads=os.cpu_count(),
                verbose=False,
            )
            loaded_model_id = selected_id
            print("Ready. Run the next cell to open the chat.")
            load_btn.description = "Loaded"
            load_btn.button_style = "success"
        except Exception as e:
            print(f"Error: {e}")
            load_btn.disabled = False
            load_btn.description = "Load Model"

load_btn.on_click(on_load)

display(model_dropdown, file_dropdown, load_btn, status)

## 4 — Launch Chat UI

In [ ]:
import json
from google.colab import output as colab_output
from IPython.display import display, HTML
from IPython.display import JSON as IPJSON

assert llm is not None, "Load a model first (cell above)."

MAX_TOKENS = 512


def generate(messages: list) -> str:
    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=0.7,
        top_k=50,
        repeat_penalty=1.05,
    )
    return response["choices"][0]["message"]["content"].strip()


def _chat_callback(messages_json):
    try:
        messages = json.loads(messages_json)
        reply = generate(messages)
        return IPJSON({"reply": reply})
    except Exception as e:
        return IPJSON({"error": str(e)})


colab_output.register_callback("notebook.chat", _chat_callback)

short_name = loaded_model_id.split("/")[-1]
quant_label = file_dropdown.value.split("/")[-1]  # last path component

CHAT_HTML = f"""
<div id="gguf-chat" style="
  font-family: 'Segoe UI', system-ui, -apple-system, sans-serif;
  max-width: 640px;
  border: 1px solid #d0d0d0;
  border-radius: 12px;
  overflow: hidden;
  background: #fafafa;
">
  <div style="
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    color: white;
    padding: 14px 18px;
    font-size: 15px;
    font-weight: 600;
    display: flex;
    align-items: center;
    gap: 8px;
  ">
    <span style="font-size: 20px;">&#x1F916;</span>
    {short_name}
    &nbsp;<span style="font-weight:400; opacity:0.7; font-size:12px;">GGUF &middot; {quant_label}</span>
  </div>

  <div id="gguf-messages" style="
    height: 380px;
    overflow-y: auto;
    padding: 16px;
    display: flex;
    flex-direction: column;
    gap: 10px;
  "></div>

  <div style="
    display: flex;
    border-top: 1px solid #e0e0e0;
    background: white;
    padding: 8px;
    gap: 8px;
  ">
    <input id="gguf-input" type="text" placeholder="Type a message..."
      style="
        flex: 1;
        border: 1px solid #d0d0d0;
        border-radius: 8px;
        padding: 10px 14px;
        font-size: 14px;
        outline: none;
      "
    />
    <button id="gguf-send" onclick="ggufSend()" style="
      background: #1a1a2e;
      color: white;
      border: none;
      border-radius: 8px;
      padding: 10px 18px;
      font-size: 14px;
      cursor: pointer;
      font-weight: 600;
    ">Send</button>
  </div>
</div>

<script>
(function() {{
  const messagesDiv = document.getElementById("gguf-messages");
  const inputEl    = document.getElementById("gguf-input");
  const sendBtn    = document.getElementById("gguf-send");
  let history = [];

  function addBubble(role, text) {{
    const isUser = role === "user";
    const bubble = document.createElement("div");
    bubble.style.cssText = `
      max-width: 82%;
      padding: 10px 14px;
      border-radius: 12px;
      font-size: 14px;
      line-height: 1.5;
      word-wrap: break-word;
      white-space: pre-wrap;
      align-self: ${{isUser ? "flex-end" : "flex-start"}};
      background: ${{isUser ? "#1a1a2e" : "#e8e8e8"}};
      color: ${{isUser ? "#fff" : "#1a1a1a"}};
    `;
    bubble.textContent = text;
    messagesDiv.appendChild(bubble);
    messagesDiv.scrollTop = messagesDiv.scrollHeight;
    return bubble;
  }}

  function extractPayload(result) {{
    if (result && result.data) {{
      if (result.data["application/json"]) {{
        const v = result.data["application/json"];
        return (typeof v === "string") ? JSON.parse(v) : v;
      }}
      if (result.data["text/plain"]) {{
        let t = result.data["text/plain"];
        t = t.replace(/^['"]/,"").replace(/['"]$/,"");
        t = t.replace(/\\'/g, "'").replace(/\\"/g, '"');
        return JSON.parse(t);
      }}
    }}
    if (result && result.reply) return result;
    throw new Error("Could not parse kernel response: " + JSON.stringify(result).substring(0, 200));
  }}

  window.ggufSend = async function() {{
    const text = inputEl.value.trim();
    if (!text) return;
    inputEl.value = "";
    inputEl.disabled = true;
    sendBtn.disabled = true;
    sendBtn.textContent = "...";

    addBubble("user", text);
    history.push({{role: "user", content: text}});
    const thinking = addBubble("assistant", "Thinking...");

    try {{
      const result = await google.colab.kernel.invokeFunction(
        "notebook.chat",
        [JSON.stringify(history)],
        {{}}
      );
      const data = extractPayload(result);
      if (data.error) throw new Error(data.error);
      thinking.textContent = data.reply;
      history.push({{role: "assistant", content: data.reply}});
    }} catch (err) {{
      thinking.textContent = "Error: " + err.message;
      thinking.style.background = "#ffe0e0";
      thinking.style.color = "#900";
      history.pop();
    }}

    inputEl.disabled = false;
    sendBtn.disabled = false;
    sendBtn.textContent = "Send";
    inputEl.focus();
  }};

  inputEl.addEventListener("keydown", (e) => {{
    if (e.key === "Enter" && !e.shiftKey) {{ e.preventDefault(); ggufSend(); }}
  }});
}})();
</script>
"""

display(HTML(CHAT_HTML))
print("Chat ready. Response time depends on model size and quant — expect 5–30 s on CPU.")

---
### Notes

**What is GGUF?**
GGUF (GPT-Generated Unified Format) is the file format used by [llama.cpp](https://github.com/ggerganov/llama.cpp). It packages weights, tokenizer, and model metadata into a single self-contained file, with built-in support for quantization at multiple precision levels. This makes it the most portable format for running models locally or on CPU — no Python ML framework required at the C++ level.

**What do the quant levels mean?**
The naming follows a pattern: `Q` = quantized, the number is bits per weight, `K` = k-quants (a more accurate quantization method), `M/S/L` = medium/small/large variant within that bit width.
- **Q2_K** — 2-bit, smallest file, noticeable quality loss
- **Q4_K_M** — 4-bit medium, best speed/quality tradeoff for most use cases
- **Q5_K_M** — 5-bit medium, closer to full quality, ~25% larger than Q4
- **Q8_0** — 8-bit, near-lossless, roughly 2× the size of Q4

The file dropdown defaults to Q4_K_M when available.

**Why CPU for GGUF?**
llama.cpp was built for CPU-first inference with BLAS acceleration (OpenBLAS, BLIS, Apple Accelerate). This notebook compiles with OpenBLAS, which vectorizes matrix multiplications across available cores. GPU support exists but requires a separate CUDA build; the other notebooks cover the GPU case via PyTorch.

**Context length (`n_ctx`)**
This sets the maximum number of tokens the model holds in its attention window — both the prompt and the generated response count toward it. 4096 is a safe default. Larger values increase RAM usage linearly; reduce it if you load a large model and hit memory limits.